# --- Preprocessing ---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import matplotlib.pyplot as plt

col_names = [
    'Team_ID', 'Jaar', 'Maandkey', 'Totaal_FTE_Ziek', 'Totaal_FTE_Toegewezen',
    'Verzuimpercentage', 'Teamomvang', 'Gem_Leeftijd', 'Aandeel_Vrouw',
    'Medewerkers_Met_Overwerk', 'Gem_Contracturen_Week', 'Aandeel_Vast_Contract',
    'Totaal_Overwerk_Uren', 'Gem_Overwerk_Per_Medewerker'
]

df = pd.read_csv("Dataset clean.csv", header=None, names=col_names, decimal=',')

df = df.sort_values(['Team_ID', 'Maandkey']).reset_index(drop=True)

df['Lag_1']  = df.groupby('Team_ID')['Verzuimpercentage'].shift(1)


df = df.dropna(subset=['Lag_1'])

features = [
    'Lag_1', 'Teamomvang', 'Gem_Leeftijd', 'Aandeel_Vrouw',
    'Gem_Contracturen_Week', 'Aandeel_Vast_Contract',
    'Gem_Overwerk_Per_Medewerker'
]

# --- Training Model ---

In [ ]:
X = df[features]
y = df['Verzuimpercentage']

train = df[df['Maandkey'] <= 202412]
val   = df[(df['Maandkey'] >= 202501) & (df['Maandkey'] <= 202512)]
test  = df[df['Maandkey'] >= 202601]

X_train, y_train = train[features], train['Verzuimpercentage']
X_val,   y_val   = val[features],   val['Verzuimpercentage']
X_test,  y_test  = test[features],  test['Verzuimpercentage']

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_val = rf.predict(X_val)

mae  = mean_absolute_error(y_val, y_pred_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
r2   = r2_score(y_val, y_pred_val)

print(f"Validation MAE:  {mae:.4f}")
print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation R²:   {r2:.4f}")

# --- Hyperparameter Tuning ---

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV MAE: {-grid_search.best_score_:.4f}")

# --- Tuned Model Training ---

In [ ]:
rf_tuned = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=4,
    min_samples_split=10,
    random_state=42
)

rf_tuned.fit(X_train, y_train)

y_pred_val = rf_tuned.predict(X_val)

mae  = mean_absolute_error(y_val, y_pred_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
r2   = r2_score(y_val, y_pred_val)

y_pred_train = rf_tuned.predict(X_train)
print(f"Train MAE:  {mean_absolute_error(y_train, y_pred_train):.4f}")
print(f"Train RMSE: {np.sqrt(mean_squared_error(y_train, y_pred_train)):.4f}")
print(f"Train R²:   {r2_score(y_train, y_pred_train):.4f}")

print(f"Tuned Validation MAE:  {mae:.4f}")
print(f"Tuned Validation RMSE: {rmse:.4f}")
print(f"Tuned Validation R²:   {r2:.4f}")

y_pred_test = rf_tuned.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2   = r2_score(y_test, y_pred_test)

print(f"Test MAE:  {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test R²:   {r2:.4f}")

# --- Baseline Model---

In [ ]:
y_baseline_test = X_test['Lag_1']

print("Baseline Test:")
print(f"  MAE:  {mean_absolute_error(y_test, y_baseline_test):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_baseline_test)):.4f}")
print(f"  R²:   {r2_score(y_test, y_baseline_test):.4f}")

# --- Residual Error Plot ---

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(y_test, y_pred_test, alpha=0.5, color='steelblue', label='RF Tuned')
ax.plot([0, 0.5], [0, 0.5], 'r--', linewidth=1.5, label='Perfect prediction')

ax.set_xlabel('Actual Absence Rate')
ax.set_ylabel('Predicted Absence Rate')
ax.set_title('Predicted vs Actual Absence Rate (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

# --- Subgroup Analysis ---

In [ ]:
test_results = X_test.copy()
test_results['Actual'] = y_test.values
test_results['Predicted'] = y_pred_test
test_results['Error'] = np.abs(test_results['Actual'] - test_results['Predicted'])

test_results['Age_Group'] = pd.cut(test_results['Gem_Leeftijd'], 
                                    bins=[0, 35, 45, 100], 
                                    labels=['<35', '35-45', '>45'])

test_results['Contract_Group'] = pd.cut(test_results['Aandeel_Vast_Contract'],
                                         bins=[0, 0.5, 1.01],
                                         labels=['<50% permanent', '>50% permanent'])

print("MAE by Age Group:")
print(test_results.groupby('Age_Group')['Error'].mean())

print("\nMAE by Contract Group:")
print(test_results.groupby('Contract_Group')['Error'].mean())

print("\nMAE by Gender (proportion female):")
test_results['Gender_Group'] = pd.cut(test_results['Aandeel_Vrouw'],
                                       bins=[0, 0.7, 1.01],
                                       labels=['<70% female', '>70% female'])
print(test_results.groupby('Gender_Group')['Error'].mean())

# --- Distribution Team Absence Rate ---

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['Verzuimpercentage'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Absence Rate')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Team Absence Rate')

axes[1].hist(df['Verzuimpercentage'], bins=30, color='steelblue', edgecolor='white', cumulative=True, density=True)
axes[1].set_xlabel('Absence Rate')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].set_title('Cumulative Distribution of Team Absence Rate')

plt.tight_layout()
plt.savefig('absence_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# --- Feature Importance ---

In [ ]:
importances = rf_tuned.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=True)
print(feat_imp)
rename = {
    'Lag_1': 'Lag-1 absence rate',
    'Teamomvang': 'Team size',
    'Gem_Leeftijd': 'Mean age',
    'Aandeel_Vrouw': 'Proportion female',
    'Gem_Contracturen_Week': 'Mean contracted hours',
    'Aandeel_Vast_Contract': 'Proportion permanent contract',
    'Gem_Overwerk_Per_Medewerker': 'Mean overtime hours',
}
feat_imp.index = [rename.get(i, i) for i in feat_imp.index]

fig, ax = plt.subplots(figsize=(8, 5))
feat_imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()